# Webscrapping+db Challenge 
**Scraping paralelo con ThreadPoolExecutor + API Open Library + SQLite**

In [1]:
# CELDA 1 - Imports y configuración inicial

import requests                             
from bs4 import BeautifulSoup               
import sqlite3                               
import time                                
import re                                    
from urllib.parse import quote               
from concurrent.futures import ThreadPoolExecutor, as_completed  
from threading import Lock                   

BASE_URL   = "https://books.toscrape.com"   
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5} 

author_cache = {}        
title_author_cache = {} 
cache_lock = Lock()      

session = requests.Session()

print("Librerías importadas correctamente")


Librerías importadas correctamente


In [2]:
# CELDA 2 - Crear la base de datos y las tablas

def crear_base_de_datos():
    conn = sqlite3.connect("books.db")
    cursor = conn.cursor()

    cursor.execute("PRAGMA foreign_keys = ON")

    # Tabla: categories 
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS categories (
            id   INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL UNIQUE
        )
    """)

    # Tabla: books
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS books (
            id             INTEGER PRIMARY KEY AUTOINCREMENT,
            title          TEXT NOT NULL,
            upc            TEXT UNIQUE,     -- Código único del producto 
            product_type   TEXT,            -- Tipo de producto 
            price_excl_tax REAL,            -- Precio sin impuesto
            price_incl_tax REAL,            -- Precio con impuesto
            tax            REAL,            -- Valor del impuesto
            availability   TEXT,            -- Disponibilidad 
            num_reviews    INTEGER,         -- Número de reseñas
            rating         INTEGER,         -- Valor 1-5
            category_id    INTEGER,         -- FK → categories.id
            book_url       TEXT,
            FOREIGN KEY (category_id) REFERENCES categories(id)
        )
    """)

    # Tabla: authors
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS authors (
            id                INTEGER PRIMARY KEY AUTOINCREMENT,
            name              TEXT NOT NULL UNIQUE,
            birth_year        INTEGER,
            country           TEXT,
            external_api_id   TEXT,
            total_known_works INTEGER,
            api_source        TEXT,
            created_at        TEXT DEFAULT (datetime('now'))  -- Fecha de inserción automática
        )
    """)

    # Tabla: book_author 
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS book_author (
            book_id   INTEGER,
            author_id INTEGER,
            PRIMARY KEY (book_id, author_id),
            FOREIGN KEY (book_id)   REFERENCES books(id),
            FOREIGN KEY (author_id) REFERENCES authors(id)
        )
    """)

    # Índices para acelerar consultas frecuentes
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_books_rating    ON books(rating)")         
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_books_price     ON books(price_excl_tax)") 
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_authors_country ON authors(country)")      

    conn.commit()  
    conn.close()   
    print("Base de datos y tablas creadas")

crear_base_de_datos()


Base de datos y tablas creadas


In [3]:
# CELDA 3 - Funciones de scraping

def obtener_categorias():
    soup = BeautifulSoup(session.get(BASE_URL, timeout=10).text, "html.parser")

    categorias = [] 

    for li in soup.find("ul", class_="nav nav-list").find_all("li")[1:]:
        a = li.find("a")  

        categorias.append((a.text.strip(), BASE_URL + "/" + a["href"]))

    print(f"{len(categorias)} categorías encontradas") 
    return categorias  


def obtener_urls_de_categoria(url_cat):
    urls   = []      
    pagina = url_cat  

    while pagina: 
        soup = BeautifulSoup(session.get(pagina, timeout=10).text, "html.parser")

        for article in soup.find_all("article", class_="product_pod"):
            href = article.h3.a["href"].replace("../../../", "catalogue/")
            urls.append(BASE_URL + "/" + href)  

        btn = soup.find("li", class_="next")
        if btn:
            base   = pagina.rsplit("/", 1)[0]  
            pagina = base + "/" + btn.find("a")["href"]  
        else:
            pagina = None  

    return urls  


def _parsear_precio(texto):
    if not texto:
        return None  

    try:
        return float(re.sub(r"[^0-9.]", "", texto))
    except ValueError:
        return None


def scrapear_libro(libro_url):
    try:
        soup = BeautifulSoup(session.get(libro_url, timeout=10).text, "html.parser")

        titulo = soup.find("h1").text.strip()

        rating_texto = soup.find("p", class_="star-rating")["class"][1]

        rating = RATING_MAP.get(rating_texto, 0)

        tabla_data = {} 
        tabla = soup.find("table", class_="table table-striped")  

        if tabla:  
            for fila in tabla.find_all("tr"):  
                th = fila.find("th")  
                td = fila.find("td")  
                if th and td:
                    tabla_data[th.text.strip()] = td.text.strip()

        upc            = tabla_data.get("UPC")                               
        product_type   = tabla_data.get("Product Type")                       
        price_excl_tax = _parsear_precio(tabla_data.get("Price (excl. tax)")) 
        price_incl_tax = _parsear_precio(tabla_data.get("Price (incl. tax)")) 
        tax            = _parsear_precio(tabla_data.get("Tax"))               
        availability   = tabla_data.get("Availability")                      
        num_reviews    = int(tabla_data.get("Number of reviews", "0") or "0")

        return {
            "titulo":         titulo,
            "rating":         rating,
            "upc":            upc,
            "product_type":   product_type,
            "price_excl_tax": price_excl_tax,
            "price_incl_tax": price_incl_tax,
            "tax":            tax,
            "availability":   availability,
            "num_reviews":    num_reviews,
            "url":            libro_url   
        }

    except Exception as e:
        print(f"Error en {libro_url}: {e}")
        return None


print("Funciones de scraping definidas")


Funciones de scraping definidas


In [4]:
# CELDA 4 - Funciones para consultar la API Open Library

def buscar_autor_por_titulo(titulo):
    with cache_lock:
        if titulo in title_author_cache:
            return title_author_cache[titulo]  

    autor_nombre = "Unknown Author" 

    try:
        query = quote(titulo[:80])

        resp  = session.get(
            f"https://openlibrary.org/search.json?title={query}&limit=1&fields=title,author_name",
            timeout=8  
        )

        if resp.status_code == 429:
            time.sleep(2)

        elif resp.status_code == 200: 
            datos = resp.json()  

            if datos.get("numFound", 0) > 0 and datos.get("docs"):
                doc     = datos["docs"][0]           
                autores = doc.get("author_name", []) 

                if autores:
                    autor_nombre = autores[0]

    except Exception:
        pass  

    with cache_lock:
        title_author_cache[titulo] = autor_nombre

    return autor_nombre  


def consultar_open_library(nombre_autor):
    if nombre_autor == "Unknown Author":
        return {
            "birth_year": None, "country": None,
            "external_api_id": None, "total_known_works": None,
            "api_source": "Open Library"
        }

    with cache_lock:
        if nombre_autor in author_cache:
            return author_cache[nombre_autor] 

    resultado = {
        "birth_year": None, "country": None,
        "external_api_id": None, "total_known_works": None,
        "api_source": "Open Library"
    }

    try:
        # PRIMERA LLAMADA: endpoint de búsqueda de autores por nombre
        resp = session.get(
            f"https://openlibrary.org/search/authors.json?q={quote(nombre_autor)}",
            timeout=8
        )

        if resp.status_code == 429:
            time.sleep(2)  

        elif resp.status_code == 200:
            datos = resp.json()

            if datos.get("numFound", 0) > 0:
                a = datos["docs"][0]  

                resultado["external_api_id"]   = a.get("key", "").replace("/authors/", "")
                resultado["total_known_works"] = a.get("work_count") 

                birth = a.get("birth_date")
                if birth and str(birth).isdigit():
                    resultado["birth_year"] = int(birth) 

                # SEGUNDA LLAMADA: perfil completo del autor para obtener su ubicación
                if resultado["external_api_id"]:
                    r2 = session.get(
                        f"https://openlibrary.org/authors/{resultado['external_api_id']}.json",
                        timeout=8
                    )
                    if r2.status_code == 200:
                        resultado["country"] = r2.json().get("location")

    except Exception:
        pass  
    with cache_lock:
        author_cache[nombre_autor] = resultado

    return resultado  


print("Funciones de API definidas")


Funciones de API definidas


In [5]:
# CELDA 5 - Funciones de inserción en la BD

def insertar_categoria(conn, nombre):
    c = conn.cursor()
    c.execute("INSERT OR IGNORE INTO categories (name) VALUES (?)", (nombre,)) 
    c.execute("SELECT id FROM categories WHERE name = ?", (nombre,))           
    return c.fetchone()[0]  


def insertar_libro(conn, datos, cat_id):
    c = conn.cursor()
    c.execute("""
        INSERT INTO books
            (title, upc, product_type, price_excl_tax, price_incl_tax,
             tax, availability, num_reviews, rating, category_id, book_url)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        datos["titulo"],        
        datos["upc"],           
        datos["product_type"],  
        datos["price_excl_tax"],
        datos["price_incl_tax"],
        datos["tax"],          
        datos["availability"],  
        datos["num_reviews"],  
        datos["rating"],        
        cat_id,                
        datos["url"]            
    ))
    return c.lastrowid  


def insertar_autor(conn, nombre, datos_api):
    c = conn.cursor()
    c.execute("""
        INSERT OR IGNORE INTO authors
            (name, birth_year, country, external_api_id, total_known_works, api_source)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        nombre,                         
        datos_api["birth_year"],        
        datos_api["country"],           
        datos_api["external_api_id"],    
        datos_api["total_known_works"], 
        datos_api["api_source"]        
    ))
    c.execute("SELECT id FROM authors WHERE name = ?", (nombre,))  
    return c.fetchone()[0]


def insertar_relacion(conn, libro_id, autor_id):
    conn.cursor().execute(
        "INSERT OR IGNORE INTO book_author (book_id, author_id) VALUES (?, ?)",
        (libro_id, autor_id)
    )


print("Funciones de inserción definidas")


Funciones de inserción definidas


In [6]:
# CELDA 6 - Scraping paralelo

MAX_WORKERS = 32  


def scrapear_todo_rapido():
    t_inicio = time.time()  

    conn = sqlite3.connect("books.db")
    conn.execute("PRAGMA foreign_keys = ON")   
    conn.execute("PRAGMA journal_mode = WAL")   
    conn.execute("PRAGMA synchronous = NORMAL") 

    categorias = obtener_categorias()  

    # PASO 1: Recolectar URLs de todos los libros 
    print("\nRecolectando URLs en paralelo...")
    todas_las_urls = []  

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futuros_cat = {
            executor.submit(obtener_urls_de_categoria, url_cat): nombre_cat
            for nombre_cat, url_cat in categorias 
        }

        for futuro in as_completed(futuros_cat):
            nombre_cat = futuros_cat[futuro] 
            urls_cat   = futuro.result()       

            for u in urls_cat:
                todas_las_urls.append((u, nombre_cat))
            print(f"  ✓ {nombre_cat}: {len(urls_cat)} libros") 

    print(f"\nTotal de libros a scrapear: {len(todas_las_urls)}")

    # PASO 2: Scrapear detalles de cada libro en paralelo 
    print(f"\nScrapeando detalles en paralelo ({MAX_WORKERS} hilos)...")
    resultados = {}  

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futuros = {executor.submit(scrapear_libro, url): url
                   for url, _ in todas_las_urls}

        completados = 0  
        for futuro in as_completed(futuros):
            url   = futuros[futuro]    
            datos = futuro.result()    

            if datos:
                resultados[url] = datos  

            completados += 1
            if completados % 100 == 0:  
                print(f"  → {completados}/{len(todas_las_urls)} libros descargados...")

    print(f"\n{len(resultados)} libros descargados.")

    # PASO 3: Buscar autor y enriquecer datos de la API 
    print(f"\nBuscando autores y enriqueciendo datos API ({MAX_WORKERS} hilos)...")

    def buscar_y_enriquecer(titulo):
        nombre    = buscar_autor_por_titulo(titulo)  
        datos_api = consultar_open_library(nombre)    
        return nombre, datos_api  

    libros_validos = list(resultados.items()) 

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futuros_autor = {
            executor.submit(buscar_y_enriquecer, datos["titulo"]): url
            for url, datos in libros_validos
        }

        completados_autor = 0
        for futuro in as_completed(futuros_autor):
            url                    = futuros_autor[futuro]  
            autor_nombre, api_data = futuro.result()         

            resultados[url]["autor"]    = autor_nombre  
            resultados[url]["api_data"] = api_data       
                                                                                                                                                                                                                                                                                                                                                                                                    
            completados_autor += 1
            if completados_autor % 100 == 0:
                print(f"  → {completados_autor}/{len(libros_validos)} autores procesados...")

    autores_encontrados = sum(
        1 for d in resultados.values() if d.get("autor") != "Unknown Author"
    )
    print(f"\nAutores encontrados: {autores_encontrados}/{len(resultados)}")

    # PASO 4: Guardar todo en la BD 
    print("\nGuardando en BD...")

    conn.execute("DELETE FROM book_author")
    conn.execute("DELETE FROM books")
    conn.execute("DELETE FROM authors")
    conn.execute("DELETE FROM categories")
    conn.execute("DELETE FROM sqlite_sequence")  
    conn.commit()

    # CATEGORÍAS:
    cats_unicas = {nombre_cat for _, nombre_cat in todas_las_urls}
    conn.executemany(
        "INSERT OR IGNORE INTO categories (name) VALUES (?)",
        [(c,) for c in cats_unicas] 
    )
    
    cat_map = dict(conn.execute("SELECT name, id FROM categories").fetchall())

    # Construimos todas las filas en memoria y las insertamos 
    filas_libros = []
    for url_libro, nombre_cat in todas_las_urls:
        datos = resultados.get(url_libro)
        if not datos:
            continue  

        filas_libros.append((
            datos["titulo"], datos["upc"], datos["product_type"],
            datos["price_excl_tax"], datos["price_incl_tax"], datos["tax"],
            datos["availability"], datos["num_reviews"], datos["rating"],
            cat_map[nombre_cat],  #
            datos["url"]
        ))

    conn.executemany("""
        INSERT OR IGNORE INTO books
            (title, upc, product_type, price_excl_tax, price_incl_tax,
             tax, availability, num_reviews, rating, category_id, book_url)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, filas_libros)

    libro_id_map = dict(conn.execute("SELECT book_url, id FROM books").fetchall())

    autores_vistos = {}  
    for url_libro, _ in todas_las_urls:
        datos = resultados.get(url_libro)
        if not datos:
            continue

        nombre = datos.get("autor", "Unknown Author")
        if nombre not in autores_vistos:
            autores_vistos[nombre] = datos.get("api_data", {
                "birth_year": None, "country": None,
                "external_api_id": None, "total_known_works": None,
                "api_source": "Open Library"
            })

    # Inserta todos los autores únicos 
    conn.executemany("""
        INSERT OR IGNORE INTO authors
            (name, birth_year, country, external_api_id, total_known_works, api_source)
        VALUES (?, ?, ?, ?, ?, ?)
    """, [
        (nombre, ad.get("birth_year"), ad.get("country"),
         ad.get("external_api_id"), ad.get("total_known_works"), ad.get("api_source"))
        for nombre, ad in autores_vistos.items()
    ])

    autor_id_map = dict(conn.execute("SELECT name, id FROM authors").fetchall())

    relaciones = []
    for url_libro, _ in todas_las_urls:
        datos = resultados.get(url_libro)
        if not datos:
            continue

        book_id   = libro_id_map.get(datos["url"])                            
        author_id = autor_id_map.get(datos.get("autor", "Unknown Author"))    

        if book_id and author_id:
            relaciones.append((book_id, author_id))  

    conn.executemany(
        "INSERT OR IGNORE INTO book_author (book_id, author_id) VALUES (?, ?)",
        relaciones
    )

    conn.commit()  
    conn.close()   

    t_total = time.time() - t_inicio  
    print(f"\n¡Listo! {len(resultados)} libros guardados en {t_total:.1f} segundos")


scrapear_todo_rapido()


50 categorías encontradas

Recolectando URLs en paralelo...
  ✓ Travel: 11 libros
  ✓ Christian Fiction: 6 libros
  ✓ Business: 12 libros
  ✓ Psychology: 7 libros
  ✓ Paranormal: 1 libros
  ✓ Parenting: 1 libros
  ✓ Philosophy: 11 libros
  ✓ Sports and Games: 5 libros
  ✓ Adult Fiction: 1 libros
  ✓ New Adult: 6 libros
  ✓ Autobiography: 9 libros
  ✓ Womens Fiction: 17 libros
  ✓ Poetry: 19 libros
  ✓ Religion: 7 libros
  ✓ Classics: 19 libros
  ✓ History: 18 libros
  ✓ Science Fiction: 16 libros
  ✓ Art: 8 libros
  ✓ Science: 14 libros
  ✓ Music: 13 libros
  ✓ Horror: 17 libros
  ✓ Humor: 10 libros
  ✓ Contemporary: 3 libros
  ✓ Biography: 5 libros
  ✓ Thriller: 11 libros
  ✓ Short Stories: 1 libros
  ✓ Politics: 3 libros
  ✓ Crime: 1 libros
  ✓ Suspense: 1 libros
  ✓ Self Help: 5 libros
  ✓ Spirituality: 6 libros
  ✓ Novels: 1 libros
  ✓ Historical: 2 libros
  ✓ Childrens: 29 libros
  ✓ Academic: 1 libros
  ✓ Erotica: 1 libros
  ✓ Cultural: 1 libros
  ✓ Christian: 3 libros
  ✓ Health

In [7]:
# CELDA 7 - Diagrama UML de la base de datos 

print("""
╔══════════════════════╗
║      categories      ║
╠══════════════════════╣
║ id (PK)              ║◄────────────────────────┐
║ name                 ║                         │
╚══════════════════════╝                         │
                                                 │
╔══════════════════════╗                         │
║        books         ║                         │
╠══════════════════════╣                         │
║ id (PK)              ║◄────────┐               │
║ title                ║         │               │
║ upc                  ║         │               │
║ product_type         ║         │               │
║ price_excl_tax       ║         │               │
║ price_incl_tax       ║         │               │
║ tax                  ║         │               │
║ availability         ║         │               │
║ num_reviews          ║         │               │
║ rating               ║         │               │
║ category_id (FK) ────╫─────────╫───────────────┘
║ book_url             ║         │
╚══════════════════════╝         │
                                 │
╔══════════════════════╗         │
║      book_author     ║         │
╠══════════════════════╣         │
║ book_id (FK, PK) ────╫─────────┘
║ author_id (FK, PK) ──╫─────────┐
╚══════════════════════╝         │
                                 │
╔══════════════════════╗         │
║        authors       ║         │
╠══════════════════════╣         │
║ id (PK)              ║◄────────┘
║ name                 ║  ← Via API /search.json?title=...
║ birth_year           ║  ← Via API /search/authors.json
║ country              ║  ← Via API /authors/{id}.json
║ external_api_id      ║  ← Via API
║ total_known_works    ║  ← Via API
║ api_source           ║
║ created_at           ║
╚══════════════════════╝
""")


╔══════════════════════╗
║      categories      ║
╠══════════════════════╣
║ id (PK)              ║◄────────────────────────┐
║ name                 ║                         │
╚══════════════════════╝                         │
                                                 │
╔══════════════════════╗                         │
║        books         ║                         │
╠══════════════════════╣                         │
║ id (PK)              ║◄────────┐               │
║ title                ║         │               │
║ upc                  ║         │               │
║ product_type         ║         │               │
║ price_excl_tax       ║         │               │
║ price_incl_tax       ║         │               │
║ tax                  ║         │               │
║ availability         ║         │               │
║ num_reviews          ║         │               │
║ rating               ║         │               │
║ category_id (FK) ────╫─────────╫───────────────┘
║ book

In [8]:
# CELDA 8 - Las 5 consultas SQL de análisis

conn   = sqlite3.connect("books.db")  
cursor = conn.cursor()

# Consulta 1: Libros bien valorados y baratos
print("=" * 60)
print("Consulta 1: Libros con +3 estrellas por debajo del precio promedio")
print("=" * 60)
cursor.execute("""
    SELECT title, price_excl_tax, rating
    FROM books
    WHERE rating > 3                          
      AND price_excl_tax < (                  
          SELECT AVG(price_excl_tax)          
          FROM books                          
      )
    ORDER BY rating DESC, price_excl_tax ASC  
    LIMIT 10                                  
""")
for f in cursor.fetchall():  
    print(f"☆{f[2]} | £{f[1]:.2f} | {f[0][:55]}")  

# Consulta 2: Categorías con mayor precio promedio + ranking 
print("\n" + "=" * 60)
print("Consulta 2: Categorías con mayor precio promedio")
print("=" * 60)
cursor.execute("""
    SELECT
        categoria,
        precio_promedio,
        RANK() OVER (ORDER BY precio_promedio DESC) AS ranking  
                                                                
    FROM (
        -- Subconsulta que genera el promedio por categoría 
        SELECT c.name AS categoria, ROUND(AVG(b.price_excl_tax), 2) AS precio_promedio
        FROM books b
        JOIN categories c ON b.category_id = c.id  
        GROUP BY c.name                            
    )
    LIMIT 5                                      
""")
for f in cursor.fetchall():
    print(f"  #{f[2]} £{f[1]} promedio | {f[0]}")  


# Consulta 3: Categorías con más libros 
print("\n" + "=" * 60)
print("Consulta 3: Top 5 categorías con más libros")
print("=" * 60)
cursor.execute("""
    SELECT c.name, COUNT(b.id) AS total
    FROM categories c
    JOIN books b ON c.id = b.category_id  
    GROUP BY c.name                      
    ORDER BY total DESC                    
    LIMIT 5
""")
for f in cursor.fetchall():
    print(f"{f[1]} libros | {f[0]}")

# Consulta 4: Distribución de ratings 
print("\n" + "=" * 60)
print("Consulta 4: Distribución de ratings")
print("=" * 60)
cursor.execute("""
    SELECT rating, COUNT(*) AS cantidad
    FROM books
    GROUP BY rating   
    ORDER BY rating DESC
""")
for f in cursor.fetchall():
    print(f"  {'☆' * f[0]} → {f[1]} libros")

# Consulta 5: País con más libros bien valorado
print("\n" + "=" * 60)
print("Consulta 5: País con más libros rating > 3")
print("=" * 60)
cursor.execute("""
    SELECT
        COALESCE(a.country, 'País desconocido') AS pais,  
        COUNT(b.id) AS total_libros,
        ROUND(AVG(b.rating), 2) AS rating_promedio
    FROM books b
    JOIN book_author ba ON b.id = ba.book_id   
    JOIN authors a      ON ba.author_id = a.id 
    WHERE b.rating > 3                          
    GROUP BY a.country                          
    ORDER BY total_libros DESC                  
    LIMIT 10
""")
for f in cursor.fetchall():
    print(f"{f[0]} | {f[1]} libros | ☆ {f[2]} promedio")

conn.close()  


Consulta 1: Libros con +3 estrellas por debajo del precio promedio
☆5 | £10.00 | An Abundance of Katherines
☆5 | £10.23 | Greek Mythic History
☆5 | £11.05 | The Power Greens Cookbook: 140 Delicious Superfood Reci
☆5 | £11.21 | Dear Mr. Knightley
☆5 | £11.33 | The Darkest Corners
☆5 | £11.38 | Naturally Lean: 125 Nourishing Gluten-Free, Plant-Based
☆5 | £11.64 | Fruits Basket, Vol. 2 (Fruits Basket #2)
☆5 | £11.83 | Old School (Diary of a Wimpy Kid #10)
☆5 | £11.89 | Superman Vol. 1: Before Truth (Superman by Gene Luen Ya
☆5 | £12.16 | Every Heart a Doorway (Every Heart A Doorway #1)

Consulta 2: Categorías con mayor precio promedio
  #1 £58.33 promedio | Suspense
  #2 £54.81 promedio | Novels
  #3 £53.61 promedio | Politics
  #4 £51.45 promedio | Health
  #5 £46.38 promedio | New Adult

Consulta 3: Top 5 categorías con más libros
152 libros | Default
110 libros | Nonfiction
75 libros | Sequential Art
67 libros | Add a comment
65 libros | Fiction

Consulta 4: Distribución de ratings
  ☆

In [9]:
# CELDA 9 - Indexación y performance 

conn   = sqlite3.connect("books.db")
cursor = conn.cursor()

cursor.execute("DROP INDEX IF EXISTS idx_price_rating")
conn.commit()

# Consulta de prueba: busca libros en rango de precio Y con rating alto
QUERY = """
    SELECT b.title, b.price_excl_tax, b.rating, c.name
    FROM books b
    JOIN categories c ON b.category_id = c.id
    WHERE b.price_excl_tax BETWEEN 10 AND 30  
      AND b.rating >= 4                        
    ORDER BY b.price_excl_tax DESC
"""

# Medición SIN índice 
inicio = time.time()
cursor.execute(QUERY)
res = cursor.fetchall()
tiempo_sin = time.time() - inicio 
print(f"Sin índice:     {tiempo_sin:.4f} segundos | {len(res)} resultados")

cursor.execute("CREATE INDEX IF NOT EXISTS idx_price_rating ON books(price_excl_tax, rating)")
conn.commit()

# Medición CON índice 
inicio = time.time()
cursor.execute(QUERY)
res2 = cursor.fetchall()
tiempo_con = time.time() - inicio
print(f"Con índice:     {tiempo_con:.4f} segundos | {len(res2)} resultados")

print("""
   ¿Por qué mejora con el índice?
   Sin índice → SQLite lee toda la tabla fila por fila (Full Table Scan)
   Con índice → SQLite salta directamente a los registros relevantes
""")

print("Plan de ejecución:")
cursor.execute("EXPLAIN QUERY PLAN " + QUERY)
for f in cursor.fetchall():
    print(f"   → {f}")  

conn.close()


Sin índice:     0.0015 segundos | 145 resultados
Con índice:     0.0009 segundos | 145 resultados

   ¿Por qué mejora con el índice?
   Sin índice → SQLite lee toda la tabla fila por fila (Full Table Scan)
   Con índice → SQLite salta directamente a los registros relevantes

Plan de ejecución:
   → (5, 0, 162, 'SEARCH b USING INDEX idx_books_price (price_excl_tax>? AND price_excl_tax<?)')
   → (13, 0, 45, 'SEARCH c USING INTEGER PRIMARY KEY (rowid=?)')


In [10]:
# CELDA 10 - Verificación final de datos en la BD

conn   = sqlite3.connect("books.db") 
cursor = conn.cursor()                

print("VERIFICACIÓN DE DATOS")
print("=" * 60)

# Lista de tuplas para mostrar estadísticas clave
for label, query in [
    ("Total libros:",                       "SELECT COUNT(*) FROM books"),

    ("Libros con UPC:",                     "SELECT COUNT(*) FROM books WHERE upc IS NOT NULL"),

    ("Libros con price_incl_tax:",          "SELECT COUNT(*) FROM books WHERE price_incl_tax IS NOT NULL"),

    ("Total autores:",                      "SELECT COUNT(*) FROM authors"),

    ("Autores CON datos de API:",           "SELECT COUNT(*) FROM authors WHERE external_api_id IS NOT NULL"),

    ("Autores SIN datos (NULL):",           "SELECT COUNT(*) FROM authors WHERE external_api_id IS NULL"),

    ("Libros con autor conocido:",
     "SELECT COUNT(*) FROM books b JOIN book_author ba ON b.id=ba.book_id "
     "JOIN authors a ON ba.author_id=a.id WHERE a.name != 'Unknown Author'"),
]:
    cursor.execute(query)   
    print(f"{label:35} {cursor.fetchone()[0]}")  

# Muestra de libros con datos completos 
print("\nMuestra de libros con datos completos:")
cursor.execute("""
    SELECT title, upc, price_excl_tax, price_incl_tax, tax, availability, num_reviews
    FROM books
    WHERE upc IS NOT NULL  
    LIMIT 5                
""")
for b in cursor.fetchall():
    # Desempaquetamos los 7 campos del SELECT en variables de posición
    print(f"{b[0][:30]:30} | UPC: {b[1]} | excl: £{b[2]} | incl: £{b[3]} "
          f"| tax: £{b[4]} | {b[5]} | reviews: {b[6]}")

# Muestra de autores enriquecidos por la API 
print("\nMuestra de autores con datos de API:")
cursor.execute("""
    SELECT name, birth_year, country, external_api_id, total_known_works
    FROM authors
    WHERE external_api_id IS NOT NULL  
    LIMIT 8                            
""")
for a in cursor.fetchall():
    print(f"{a[0][:28]:28} | Nacim: {a[1]} | País: {a[2]} | Obras: {a[4]}")

conn.close()


VERIFICACIÓN DE DATOS
Total libros:                       1000
Libros con UPC:                     1000
Libros con price_incl_tax:          1000
Total autores:                      19
Autores CON datos de API:           10
Autores SIN datos (NULL):           9
Libros con autor conocido:          19

Muestra de libros con datos completos:
It's Only the Himalayas        | UPC: a22124811bfa8350 | excl: £45.17 | incl: £45.17 | tax: £0.0 | In stock (19 available) | reviews: 0
Full Moon over Noahâs Ark: A | UPC: ce60436f52c5ee68 | excl: £49.43 | incl: £49.43 | tax: £0.0 | In stock (15 available) | reviews: 0
See America: A Celebration of  | UPC: f9705c362f070608 | excl: £48.87 | incl: £48.87 | tax: £0.0 | In stock (14 available) | reviews: 0
Vagabonding: An Uncommon Guide | UPC: 1809259a5a5f1d8d | excl: £36.94 | incl: £36.94 | tax: £0.0 | In stock (8 available) | reviews: 0
Under the Tuscan Sun           | UPC: a94350ee74deaa07 | excl: £37.33 | incl: £37.33 | tax: £0.0 | In stock (7 availa